<a href="https://colab.research.google.com/github/Adlite07/nlp-financial-risk-early-warning/blob/main/notebooks/01_stage1_news_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path                # Path makes file paths safe across operating systems

# Root folder for the whole project, inside your Drive
ROOT = Path('/content/drive/MyDrive/risk-nlp-project')

# One folder per purpose keeps the project tidy, and mirrors the GitHub repo layout later
for sub in ['data/raw', 'data/processed', 'models', 'results', 'notebooks', 'src']:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)   # exist_ok=True: no error if it already exists

print(sorted(p.relative_to(ROOT).as_posix() for p in ROOT.rglob('*')))  # list what we made

['data', 'data/processed', 'data/raw', 'data/raw/phrasebank_50agree.csv', 'models', 'notebooks', 'results', 'results/environment.json', 'src']


In [ ]:
import pandas as pd

# Path to raw data folder
RAW_DATA_DIR = ROOT / 'data/raw'

# List all files currently inside the raw data directory
raw_files = list(RAW_DATA_DIR.glob('*'))
print(f"Files in raw directory: {[f.name for f in raw_files]}")

# Helper Template: Uncomment and modify the lines below to load your specific dataset when uploaded
# if raw_files:
#     # Example for loading the first CSV found in your raw directory
#     csv_files = list(RAW_DATA_DIR.glob('*.csv'))
#     if csv_files:
#         df = pd.read_csv(csv_files[0])
#         display(df.head())
# else:
#     print("Please place your dataset files inside:", RAW_DATA_DIR)

In [3]:
# "<4.0" means any version below 4.0, which still allows dataset scripts
!pip install -q "datasets<4.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [4]:
import sys, torch, sklearn, pandas, numpy
import transformers, datasets

# Record the exact versions: different versions can give slightly different results
print('Python      :', sys.version.split()[0])
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets    :', datasets.__version__)
print('sklearn     :', sklearn.__version__)
print('pandas      :', pandas.__version__)

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name     :', torch.cuda.get_device_name(0))

Python      : 3.13.16
torch       : 2.14.1+cu130
transformers: 5.22.1
datasets    : 3.6.0
sklearn     : 1.6.1
pandas      : 2.2.3
GPU available: True
GPU name     : Tesla T4


In [5]:
import random, numpy as np, torch

SEED = 42   # any number works; what matters is that you never change it between runs

def set_seed(seed: int = SEED):
    random.seed(seed)                    # Python's built-in random module
    np.random.seed(seed)                 # NumPy (used by scikit-learn too)
    torch.manual_seed(seed)              # PyTorch on CPU
    torch.cuda.manual_seed_all(seed)     # PyTorch on all GPUs

set_seed()
print(np.random.rand(3))   # run this cell twice: you should get identical numbers each time

[0.37454012 0.95071431 0.73199394]


In [6]:
import json, platform

env = {
    'seed': SEED,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'sklearn': sklearn.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
with open(ROOT / 'results' / 'environment.json', 'w') as f:
    json.dump(env, f, indent=2)          # indent=2 makes the file readable
print('Saved to', ROOT / 'results' / 'environment.json')

Saved to /content/drive/MyDrive/risk-nlp-project/results/environment.json


In [7]:
import datasets
print(datasets.__version__)   # should print a number starting with 3

3.6.0


In [8]:
from huggingface_hub import list_repo_files

# Ask the Hub for the file list of the namespaced repo. This does not download the data.
try:
    files = list_repo_files("takala/financial_phrasebank", repo_type="dataset")
    for f in files:
        print(f)
except Exception as e:
    print("LIST FAILED:", type(e).__name__, str(e)[:300])

.gitattributes
README.md
data/FinancialPhraseBank-v1.0.zip
financial_phrasebank.py


In [9]:
from huggingface_hub import hf_hub_download
import zipfile

# Download the zip from the dataset repo. The filename is exactly as in your listing.
zip_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    filename="data/FinancialPhraseBank-v1.0.zip",
    repo_type="dataset",
)
print("Downloaded to:", zip_path)

# Look inside the zip without extracting everything
with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        print(name)

data/FinancialPhraseBank-v1.0.zip: reconstructing file:   0%|          |  0.00B /  682kB            

data/FinancialPhraseBank-v1.0.zip: downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/datasets--takala--financial_phrasebank/snapshots/8d3fe0c36d5feec6b3cc5e455b0fcb4820fb9964/data/FinancialPhraseBank-v1.0.zip
FinancialPhraseBank-v1.0/
FinancialPhraseBank-v1.0/License.txt
__MACOSX/
__MACOSX/FinancialPhraseBank-v1.0/
__MACOSX/FinancialPhraseBank-v1.0/._License.txt
FinancialPhraseBank-v1.0/README.txt
__MACOSX/FinancialPhraseBank-v1.0/._README.txt
FinancialPhraseBank-v1.0/Sentences_50Agree.txt
FinancialPhraseBank-v1.0/Sentences_66Agree.txt
FinancialPhraseBank-v1.0/Sentences_75Agree.txt
FinancialPhraseBank-v1.0/Sentences_AllAgree.txt


In [10]:
# Paste the exact path of the 50%-agreement file from the Cell 1 output
TARGET = "FinancialPhraseBank-v1.0/Sentences_50Agree.txt"   # CHECK this against your listing

with zipfile.ZipFile(zip_path) as z:
    raw = z.read(TARGET)                      # read the file as raw bytes

# Try UTF-8 first; fall back to latin-1 (an older encoding) if that fails
try:
    text = raw.decode("utf-8")
    print("Decoded as utf-8")
except UnicodeDecodeError:
    text = raw.decode("latin-1")
    print("Decoded as latin-1")

lines = text.splitlines()                      # one sentence per line
print("Number of lines:", len(lines))
for line in lines[:5]:                         # show the first five lines exactly as they are
    print(repr(line))                          # repr shows hidden characters

Decoded as latin-1
Number of lines: 4846
'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .@neutral'
'Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .@neutral'
'The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .@negative'
'With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .@positive'
"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating 

In [11]:
import pandas as pd

rows = []
for line in lines:
    line = line.strip()
    if not line:
        continue                                    # skip blank lines
    # rsplit from the right, once: a sentence might contain '@' itself,
    # but the label is always after the LAST '@'
    sentence, label_word = line.rsplit("@", 1)
    rows.append((sentence.strip(), label_word.strip().lower()))

df = pd.DataFrame(rows, columns=["sentence", "label_name"])

# Check that only the three expected labels appear (catches parsing mistakes)
print(df["label_name"].value_counts())

# Our own numeric labels, fixed here so they never change later
label_names = ["negative", "neutral", "positive"]
df["label"] = df["label_name"].map({n: i for i, n in enumerate(label_names)})
assert df["label"].notna().all(), "Unexpected label word found"   # stops if parsing went wrong
print(df.shape)

label_name
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
(4846, 3)


In [12]:
# Class balance as percentages
print((df["label_name"].value_counts() / len(df) * 100).round(1))

# Sentence length in words
df["n_words"] = df["sentence"].str.split().str.len()
print(df["n_words"].describe())

# Duplicates: identical sentences could end up in both train and test (leakage)
print("Duplicate sentences:", df["sentence"].duplicated().sum())

# Three random examples per class: read them, numbers alone can mislead
for lab in label_names:
    print(f"\n--- {lab} ---")
    for s in df[df["label_name"] == lab]["sentence"].sample(3, random_state=42):
        print("*", s)

# Save the raw table to Drive so we never depend on the loader again
df[["sentence", "label", "label_name"]].to_csv(
    ROOT / "data" / "raw" / "phrasebank_50agree.csv", index=False)
print("Saved")

label_name
neutral     59.4
positive    28.1
negative    12.5
Name: count, dtype: float64
count    4846.000000
mean       23.101114
std         9.958474
min         2.000000
25%        16.000000
50%        21.000000
75%        29.000000
max        81.000000
Name: n_words, dtype: float64
Duplicate sentences: 8

--- negative ---
* The company decided at the end of 2008 to temporarily shut down its ammonia plant in Billingham and extend the maintenance period at its Ince facility .
* down to EUR5 .9 m H1 '09 3 August 2009 - Finnish media group Ilkka-Yhtyma Oyj ( HEL : ILK2S ) said today its net profit fell 45 % on the year to EUR5 .9 m in the first half of 2009 .
* The steelmaker said that the drop in profit was explained by the continuing economic uncertainty , mixed with the current drought in bank lending , resulting in a decline in demand for its products as customers find it increasingly difficult to fund operations .

--- neutral ---
* The center offers a comprehensive range of devi

In [13]:
# Find all rows whose sentence appears more than once
dups = df[df["sentence"].duplicated(keep=False)].sort_values("sentence")
print(dups[["sentence", "label_name"]].to_string())

# For each duplicated sentence, count how many different labels it has
conflict = dups.groupby("sentence")["label_name"].nunique()
print("\nSentences with conflicting labels:", (conflict > 1).sum())

                                                                                                                                                                                                                                                                                                      sentence label_name
2395                                                                                                                                                                                                                                                  Ahlstrom 's share is quoted on the NASDAQ OMX Helsinki .    neutral
2396                                                                                                                                                                                                                                                  Ahlstrom 's share is quoted on the NASDAQ OMX Helsinki .    neutral
3093                                                      

In [15]:

import pandas as pd

# Reload from Drive so this step does not depend on earlier cells
df = pd.read_csv(ROOT / "data" / "raw" / "phrasebank_50agree.csv")
print("Before cleaning:", df.shape)

# Sentences that appear with more than one distinct label = conflicting
n_labels = df.groupby("sentence")["label"].transform("nunique")
df = df[n_labels == 1]                                  # drop ALL copies of conflicting sentences

# Remaining exact duplicates share a label, so keep the first copy only
df = df.drop_duplicates(subset="sentence", keep="first").reset_index(drop=True)
print("After cleaning:", df.shape)                      # expect (4836, 3)

Before cleaning: (4846, 3)
After cleaning: (4836, 3)


In [16]:
# Normalise: lowercase and collapse whitespace, then look for repeats
norm = df["sentence"].str.lower().str.split().str.join(" ")
print("Near-duplicates remaining:", norm.duplicated().sum())   # ideally 0; tell me if not

Near-duplicates remaining: 0


In [17]:
from sklearn.model_selection import train_test_split

# Step 1: hold out 30% of the data, keeping class proportions (stratify)
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED)

# Step 2: split the held-out 30% in half: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part))
    print((part["label_name"].value_counts(normalize=True) * 100).round(1).to_dict())

train 3385
{'neutral': 59.4, 'positive': 28.2, 'negative': 12.5}
val 725
{'neutral': 59.4, 'positive': 28.1, 'negative': 12.4}
test 726
{'neutral': 59.4, 'positive': 28.1, 'negative': 12.5}


In [18]:
# The three parts must share no sentences at all
assert not set(train_df["sentence"]) & set(val_df["sentence"])
assert not set(train_df["sentence"]) & set(test_df["sentence"])
assert not set(val_df["sentence"]) & set(test_df["sentence"])
print("No overlap between splits")

# Save each split so every later step uses exactly the same data
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.to_csv(ROOT / "data" / "processed" / f"phrasebank_{name}.csv", index=False)
print("Saved")

No overlap between splits
Saved


In [19]:
import pandas as pd

# Reload the three saved splits from Drive (this also confirms they were saved)
parts = {n: pd.read_csv(ROOT / "data" / "processed" / f"phrasebank_{n}.csv")
         for n in ["train", "val", "test"]}

total = sum(len(p) for p in parts.values())
print("Total rows across splits:", total)             # expect 4836

for name, p in parts.items():
    pct = (p["label_name"].value_counts(normalize=True) * 100).round(1).to_dict()
    n_neg = (p["label_name"] == "negative").sum()    # how many negatives each split has
    print(name, len(p), pct, "| negatives:", n_neg)

Total rows across splits: 4836
train 3385 {'neutral': 59.4, 'positive': 28.2, 'negative': 12.5} | negatives: 423
val 725 {'neutral': 59.4, 'positive': 28.1, 'negative': 12.4} | negatives: 90
test 726 {'neutral': 59.4, 'positive': 28.1, 'negative': 12.5} | negatives: 91


In [20]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, random
ROOT = Path('/content/drive/MyDrive/risk-nlp-project')
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Load the saved splits
train_df = pd.read_csv(ROOT / "data/processed/phrasebank_train.csv")
val_df   = pd.read_csv(ROOT / "data/processed/phrasebank_val.csv")
label_names = ["negative", "neutral", "positive"]   # 0, 1, 2: fixed in Step 1.1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
from sklearn.dummy import DummyClassifier   # sklearn's built-in "dumb" model for baselines

# strategy="most_frequent": always predict the most common training class
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train_df[["sentence"]], train_df["label"])   # it only looks at the labels

val_pred = baseline.predict(val_df[["sentence"]])
print("Unique predictions:", np.unique(val_pred))          # expect just [1] = neutral

Unique predictions: [1]


In [22]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", round(accuracy_score(val_df["label"], val_pred), 3))

# zero_division=0 stops warnings when a class is never predicted (precision undefined)
print(classification_report(val_df["label"], val_pred,
                            target_names=label_names, digits=3, zero_division=0))

cm = confusion_matrix(val_df["label"], val_pred, labels=[0, 1, 2])
print(pd.DataFrame(cm, index=[f"true_{n}" for n in label_names],
                      columns=[f"pred_{n}" for n in label_names]))

Accuracy: 0.594
              precision    recall  f1-score   support

    negative      0.000     0.000     0.000        90
     neutral      0.594     1.000     0.746       431
    positive      0.000     0.000     0.000       204

    accuracy                          0.594       725
   macro avg      0.198     0.333     0.249       725
weighted avg      0.353     0.594     0.443       725

               pred_negative  pred_neutral  pred_positive
true_negative              0            90              0
true_neutral               0           431              0
true_positive              0           204              0


In [23]:
from sklearn.metrics import recall_score, f1_score
import json

results = {
    "model": "majority_baseline", "split": "validation",
    "accuracy": float(accuracy_score(val_df["label"], val_pred)),
    "negative_recall": float(recall_score(val_df["label"], val_pred, labels=[0], average=None)[0]),
    "macro_f1": float(f1_score(val_df["label"], val_pred, average="macro")),
}
with open(ROOT / "results" / "baseline_majority_val.json", "w") as f:
    json.dump(results, f, indent=2)
print(results)

{'model': 'majority_baseline', 'split': 'validation', 'accuracy': 0.5944827586206897, 'negative_recall': 0.0, 'macro_f1': 0.248558246828143}


In [24]:
#tf-idf
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, f1_score, accuracy_score

def make_pipe(ngram, C, cw):
    return Pipeline([
        # lowercase=True: "Profit" and "profit" count as the same word
        # min_df=2: ignore words seen in fewer than 2 training sentences (keeps memory low)
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=ngram, min_df=2)),
        # max_iter=1000: allow enough steps to converge; random_state fixes the seed
        ("clf", LogisticRegression(C=C, class_weight=cw, max_iter=1000, random_state=SEED)),
    ])

pipe = make_pipe(ngram=(1, 2), C=1.0, cw="balanced")
pipe.fit(train_df["sentence"], train_df["label"])     # vocabulary learned from TRAIN only
pred = pipe.predict(val_df["sentence"])

print("accuracy       :", round(accuracy_score(val_df["label"], pred), 3))
print("negative recall:", round(recall_score(val_df["label"], pred, labels=[0], average=None)[0], 3))
print("macro-F1       :", round(f1_score(val_df["label"], pred, average="macro"), 3))

accuracy       : 0.745
negative recall: 0.578
macro-F1       : 0.683


In [25]:
import itertools

rows = []
# 2 n-gram settings x 3 C values x 2 weight settings = 12 models, quick to train
for ngram, C, cw in itertools.product([(1, 1), (1, 2)], [0.1, 1, 10], [None, "balanced"]):
    p = make_pipe(ngram, C, cw).fit(train_df["sentence"], train_df["label"])
    pr = p.predict(val_df["sentence"])
    rows.append({
        "ngram": ngram, "C": C, "class_weight": cw,
        "accuracy": accuracy_score(val_df["label"], pr),
        "neg_recall": recall_score(val_df["label"], pr, labels=[0], average=None)[0],
        "macro_f1": f1_score(val_df["label"], pr, average="macro"),
    })

grid = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).round(3)
print(grid.to_string(index=False))

 ngram    C class_weight  accuracy  neg_recall  macro_f1
(1, 1) 10.0     balanced     0.760       0.589     0.701
(1, 1)  1.0     balanced     0.749       0.589     0.689
(1, 2) 10.0     balanced     0.752       0.533     0.685
(1, 1) 10.0         None     0.763       0.456     0.684
(1, 2)  1.0     balanced     0.745       0.578     0.683
(1, 2) 10.0         None     0.757       0.444     0.678
(1, 2)  0.1     balanced     0.735       0.511     0.655
(1, 1)  1.0         None     0.752       0.356     0.648
(1, 1)  0.1     balanced     0.723       0.533     0.645
(1, 2)  1.0         None     0.753       0.322     0.639
(1, 1)  0.1         None     0.659       0.078     0.419
(1, 2)  0.1         None     0.644       0.067     0.389


In [26]:
from sklearn.metrics import classification_report, confusion_matrix

best = grid.iloc[0]                                    # top row = best validation macro-F1
best_ngram, best_C, best_cw = best["ngram"], best["C"], best["class_weight"]
print("Chosen:", best_ngram, best_C, best_cw)

final = make_pipe(best_ngram, best_C, best_cw).fit(train_df["sentence"], train_df["label"])
pred = final.predict(val_df["sentence"])

print(classification_report(val_df["label"], pred, target_names=label_names, digits=3))
cm = confusion_matrix(val_df["label"], pred, labels=[0, 1, 2])
print(pd.DataFrame(cm, index=[f"true_{n}" for n in label_names],
                       columns=[f"pred_{n}" for n in label_names]))

Chosen: (1, 1) 10.0 balanced
              precision    recall  f1-score   support

    negative      0.602     0.589     0.596        90
     neutral      0.832     0.838     0.835       431
    positive      0.675     0.672     0.673       204

    accuracy                          0.760       725
   macro avg      0.703     0.699     0.701       725
weighted avg      0.759     0.760     0.760       725

               pred_negative  pred_neutral  pred_positive
true_negative             53            23             14
true_neutral              18           361             52
true_positive             17            50            137


In [27]:
vec = final.named_steps["tfidf"]
clf = final.named_steps["clf"]
words = np.array(vec.get_feature_names_out())
neg_weights = clf.coef_[0]                 # row 0 = class 0 = negative
top = np.argsort(neg_weights)[-15:][::-1]  # 15 largest weights
print("Words pushing toward NEGATIVE:", list(words[top]))

Words pushing toward NEGATIVE: ['down', 'decreased', 'fell', 'result', 'dropped', 'staff', 'lower', 'compared', 'lay', 'negative', 'cut', 'news', 'left', 'declined', 'canceled']


In [28]:
import json, joblib
res = {"model": "tfidf_logreg", "split": "validation",
       "ngram": list(best_ngram), "C": float(best_C), "class_weight": str(best_cw),
       "accuracy": float(accuracy_score(val_df["label"], pred)),
       "negative_recall": float(recall_score(val_df["label"], pred, labels=[0], average=None)[0]),
       "macro_f1": float(f1_score(val_df["label"], pred, average="macro"))}
with open(ROOT / "results/tfidf_logreg_val.json", "w") as f:
    json.dump(res, f, indent=2)
grid.to_csv(ROOT / "results/tfidf_logreg_grid.csv", index=False)   # keep the full grid as evidence
joblib.dump(final, ROOT / "models/tfidf_logreg.joblib")            # models/ is git-ignored
print(res)

{'model': 'tfidf_logreg', 'split': 'validation', 'ngram': [1, 1], 'C': 10.0, 'class_weight': 'balanced', 'accuracy': 0.76, 'negative_recall': 0.5888888888888889, 'macro_f1': 0.7011354573736855}


In [29]:
#finbert
import torch, numpy as np, pandas as pd, random, json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
dev = "cuda"
tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(dev)

print(model.config.id2label)   # CHECK: FinBERT's own label order (I expect positive/negative/neutral)
# Convert OUR labels (0 neg,1 neu,2 pos) into FinBERT's order so its pretrained head stays valid
f2i = {v.lower(): k for k, v in model.config.id2label.items()}
conv = {0: f2i["negative"], 1: f2i["neutral"], 2: f2i["positive"]}
NEG = f2i["negative"]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{0: 'positive', 1: 'negative', 2: 'neutral'}


In [30]:
val_df  = pd.read_csv(ROOT/"data/processed/phrasebank_val.csv")
train_df = pd.read_csv(ROOT/"data/processed/phrasebank_train.csv")
ytr = train_df["label"].map(conv).values; yva = val_df["label"].map(conv).values

def encode(texts):   # turn sentences into token ids, padded/truncated to 128
    return tok(list(texts), truncation=True, max_length=128, padding=True, return_tensors="pt")

# Balanced class weights, same idea as logistic regression: rare class errors cost more
counts = np.bincount(ytr, minlength=3)
w = torch.tensor(len(ytr) / (3 * counts), dtype=torch.float).to(dev)
loss_fn = torch.nn.CrossEntropyLoss(weight=w)

In [31]:
from sklearn.metrics import f1_score, recall_score
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()          # keeps 16-bit training stable
BS, EPOCHS = 16, 3
best_f1 = -1

def predict(texts, bs=64):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            b = {k: v.to(dev) for k, v in encode(texts[i:i+bs]).items()}
            with torch.autocast("cuda", dtype=torch.float16):
                out.append(model(**b).logits.argmax(-1).cpu())
    return torch.cat(out).numpy()

texts = train_df["sentence"].tolist()
for ep in range(EPOCHS):
    model.train()
    idx = np.random.permutation(len(texts))   # reshuffle each epoch (seeded)
    for i in range(0, len(idx), BS):
        j = idx[i:i+BS]
        b = {k: v.to(dev) for k, v in encode([texts[k] for k in j]).items()}
        y = torch.tensor(ytr[j]).to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(**b).logits.float(), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    p = predict(val_df["sentence"].tolist())
    f1 = f1_score(yva, p, average="macro")
    nr = recall_score(yva, p, labels=[NEG], average=None)[0]
    print(f"epoch {ep+1}: val macro-F1 {f1:.3f}, negative recall {nr:.3f}")
    if f1 > best_f1:                           # save the best epoch (chosen on VALIDATION)
        best_f1 = f1
        model.save_pretrained(ROOT/"models/finbert_ft"); tok.save_pretrained(ROOT/"models/finbert_ft")

/tmp/ipykernel_425/2984549328.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()          # keeps 16-bit training stable


epoch 1: val macro-F1 0.840, negative recall 1.000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 2: val macro-F1 0.846, negative recall 0.911


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

epoch 3: val macro-F1 0.868, negative recall 0.933


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [32]:
import joblib, json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score, accuracy_score

test_df = pd.read_csv(ROOT/"data/processed/phrasebank_test.csv")
label_names = ["negative", "neutral", "positive"]
y_true = test_df["label"].values

def finbert_predict(path, texts, bs=64):
    # Load a FinBERT model, predict in batches, and return OUR label codes (0 neg, 1 neu, 2 pos)
    tk = AutoTokenizer.from_pretrained(path)
    m = AutoModelForSequenceClassification.from_pretrained(path).to("cuda").eval()
    f2i = {v.lower(): k for k, v in m.config.id2label.items()}
    back = {f2i["negative"]: 0, f2i["neutral"]: 1, f2i["positive"]: 2}   # FinBERT code -> our code
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            b = tk(texts[i:i+bs], truncation=True, max_length=128, padding=True, return_tensors="pt").to("cuda")
            out.append(m(**b).logits.argmax(-1).cpu())
    return np.array([back[int(p)] for p in torch.cat(out)])

def report(name, y, p):
    # Print the key numbers and a bootstrap 95% interval for negative recall
    rng = np.random.default_rng(42)                      # fixed seed for the resampling
    boots = []
    for _ in range(1000):
        i = rng.integers(0, len(y), len(y))              # resample test sentences with replacement
        boots.append(recall_score(y[i], p[i], labels=[0], average=None)[0])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"\n=== {name} ===")
    print(f"accuracy {accuracy_score(y,p):.3f} | macro-F1 {f1_score(y,p,average='macro'):.3f} | "
          f"negative recall {recall_score(y,p,labels=[0],average=None)[0]:.3f} (95% CI {lo:.3f} to {hi:.3f})")
    print(classification_report(y, p, target_names=label_names, digits=3))
    print(pd.DataFrame(confusion_matrix(y, p, labels=[0,1,2]),
          index=[f"true_{n}" for n in label_names], columns=[f"pred_{n}" for n in label_names]))

In [33]:
texts = test_df["sentence"].tolist()
tfidf = joblib.load(ROOT/"models/tfidf_logreg.joblib")
p_tfidf = tfidf.predict(test_df["sentence"])
p_ft    = finbert_predict(str(ROOT/"models/finbert_ft"), texts)
p_zero  = finbert_predict("ProsusAI/finbert", texts)      # original FinBERT, NO fine-tuning by us

report("TF-IDF + LogReg", y_true, p_tfidf)
report("FinBERT fine-tuned", y_true, p_ft)
report("FinBERT original (leakage check)", y_true, p_zero)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== TF-IDF + LogReg ===
accuracy 0.762 | macro-F1 0.736 | negative recall 0.736 (95% CI 0.636 to 0.826)
              precision    recall  f1-score   support

    negative      0.728     0.736     0.732        91
     neutral      0.833     0.810     0.821       431
    positive      0.637     0.672     0.654       204

    accuracy                          0.762       726
   macro avg      0.733     0.739     0.736       726
weighted avg      0.765     0.762     0.763       726

               pred_negative  pred_neutral  pred_positive
true_negative             67            13             11
true_neutral              15           349             67
true_positive             10            57            137

=== FinBERT fine-tuned ===
accuracy 0.883 | macro-F1 0.871 | negative recall 0.978 (95% CI 0.943 to 1.000)
              precision    recall  f1-score   support

    negative      0.788     0.978     0.873        91
     neutral      0.923     0.896     0.909       431
    positiv

In [34]:
test_df["pred_tfidf"] = p_tfidf; test_df["pred_ft"] = p_ft
missed = test_df[(test_df["label"] == 0) & (test_df["pred_ft"] != 0)]
print("FinBERT missed negatives:", len(missed))
for _, r in missed.head(10).iterrows():
    print(f"- [{label_names[r.pred_ft]}] {r.sentence}")
both = test_df[(test_df["label"] == 0) & (test_df["pred_tfidf"] != 0)]
print("\nTF-IDF missed negatives:", len(both))
for _, r in both.sample(min(10, len(both)), random_state=42).iterrows():
    print(f"- [{label_names[r.pred_tfidf]}] {r.sentence}")
test_df.to_csv(ROOT/"results/test_predictions.csv", index=False)

FinBERT missed negatives: 2
- [neutral] A tinyurl link takes users to a scamming site promising that users can earn thousands of dollars by becoming a Google ( NASDAQ : GOOG ) Cash advertiser .
- [neutral] The reduction notice applied to 17 people .

TF-IDF missed negatives: 24
- [neutral] Vanhanen said the strike would be `` extremely damaging '' as some 1,300 participants and reporters begin to arrive in Finland for the one-day EU summit with Russian President Vladimir Putin in Lahti , about 100 kilometers ( 60 miles ) north of Helsinki .
- [positive] Diluted earnings per share ( EPS ) stood at EUR 0.25 versus EUR 0.42 .
- [positive] The result will also be burdened by increased fixed costs associated with operations in China , and restructuring costs in Japan .
- [positive] He said he has been losing five families a month to the economy as many make alternative arrangements to save money .
- [neutral] Finnish meat company Atria can no longer promise a sufficient amount of domestic b

In [14]:
from datasets import load_dataset

try:
    # Same call as before, but with the full namespace/name
    ds = load_dataset("takala/financial_phrasebank", "sentences_50agree", trust_remote_code=True)
    print(ds)
except Exception as e:
    print("LOAD FAILED:", type(e).__name__)
    print(str(e)[:800])

README.md:   0%|          | 0.00/9.58k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

LOAD FAILED: HfUriError
Invalid HF URI 'hf://datasets/financial_phrasebank@main/data/FinancialPhraseBank-v1.0.zip'. Repository id must be 'namespace/name', got 'financial_phrasebank'.
